# 114. 二叉树展开为链表

给你二叉树的根结点 root ，请你将它展开为一个单链表：

- 展开后的单链表应该同样使用 TreeNode ，其中 right 子指针指向链表中下一个结点，而左子指针始终为 null 。
- 展开后的单链表应该与二叉树 先序遍历 顺序相同。

示例 1：

![example1](https://assets.leetcode.com/uploads/2021/01/14/flaten.jpg)

> 输入：root = [1,2,5,3,4,null,6]
> 
> 输出：[1,null,2,null,3,null,4,null,5,null,6]

示例 2：

> 输入：root = []
> 
> 输出：[]

示例 3：

> 输入：root = [0]
> 
> 输出：[0]

## 简单思路

先序遍历树，将结果存到数组里，然后再创建链表，左指针None，右指针指向新节点。

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def flatten(self, root: Optional[TreeNode]) -> None:
        """
        Do not return anything, modify root in-place instead.
        """
        # 简单做法，先先序遍历，然后在创建链表
        nums = []
        def first_travel(root):
            if not root:
                return
            nums.append(root.val)
            first_travel(root.left)
            first_travel(root.right)
        
        first_travel(root)

        p = root
        for i in range(1, len(nums)):
            t = TreeNode(val=nums[i])
            p.right = t
            p.left = None
            p = t

## 迭代法寻找前驱节点

核心逻辑：

利用先序遍历的性质，根左右，对于当前节点 curr：

- 如果它有左子树，那么它左子树的最右侧节点，一定是它原本右子树在展开后的前驱节点。
- 所以，我们把 curr 的右子树，直接挂到 curr 左子树的最右侧节点的 right 上。
- 然后把 curr 的左子树移到右边，左边置为空。
- 继续处理下一个节点。

**思路详解**：

要求我们将二叉树按照“先序遍历（根 -> 左 -> 右）”的顺序展开成单链表。想象一下先序遍历的过程：当我们站在某个节点（比如 curr）时，我们一定是先把它的整个左子树全部遍历完，然后才去遍历它的右子树。

那么问题来了：左子树中最后一个被遍历到的节点是谁？
答案是：左子树中最右下角的那个节点。

既然这个“最右下角的节点”是左子树的终点，而 curr 的右子树是下一步的起点，我们是不是可以直接把 curr 的右子树，强行“嫁接”到这个终点节点的后面？由此引出了我们的核心算法。

**算法执行步骤**：

对于树上的任意一个当前节点 curr，只要它有左子树，我们就对它执行以下“外科手术”：

1. 找前驱： 深入 curr 的左子树，一路向右到底，找到最右侧的节点，我们称之为 predecessor（前驱节点）。

2. 接右臂： 将 curr 原本的整个右子树，直接挂到 predecessor 的右指针上。

3. 左换右： 将 curr 的整个左子树移动到右边，然后把左指针置为 null。

完成这三步后，curr 的左边就空了，而所有节点都按正确的先序顺序排在了右边。然后，我们将 curr 往右移动一步，继续重复这个过程。

图解推演：

我们用示例 1 的树来一步步推演，这棵树长这样：

```text
      1
     / \
    2   5
   / \   \
  3   4   6
```
初始状态： curr 指向根节点 1。

第一轮：

curr 是 1，它有左子树 2。

步骤 1（找前驱）： 在 1 的左子树中一路向右，找到的最右节点是 4。

步骤 2（接右臂）： 把 1 的右子树（以 5 为根）整个切下来，接到 4 的右边。

```text
      1
     /
    2
   / \
  3   4
       \
        5
         \
          6
```

步骤 3（左换右）： 把 1 的左子树（以 2 为根）移到右边，左边置空。

```text
      1
       \
        2
       / \
      3   4
           \
            5
             \
              6
```
curr 向下走一步，变成 2。

第二轮：
curr 是 2，它有左子树 3。

步骤 1（找前驱）： 在 2 的左子树中向右找，因为 3 没有右孩子，所以最右节点就是 3 本身。

步骤 2（接右臂）： 把 2 当前的右子树（以 4 为根的一大串）接到 3 的右边。

步骤 3（左换右）： 把 2 的左子树（3）移到右边。

```text
      1
       \
        2
         \
          3
           \
            4
             \
              5
               \
                6
```
curr 继续向右走一步，变成 3。

后续回合：
curr 走到 3，没有左子树，跳过，curr 走到 4。

curr 走到 4，没有左子树，跳过，curr 走到 5。

直到 curr 走到整条链表的末尾（null），循环结束。

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def flatten(self, root: Optional[TreeNode]) -> None:
        """
        Do not return anything, modify root in-place instead.
        """
        curr = root
        while curr:
            if curr.left:
                # 如果有左子树，找左子树最右侧节点
                t = curr.left
                while t.right:
                    t = t.right
                # 右子树移接
                t.right = curr.right
                # 左子树移接
                curr.right = curr.left
                curr.left = None
            # 处理下一个节点
            curr = curr.right